## Convert the FCC pdf file into a CSV sentence-delimited file

In [32]:
%pip install pandas langdetect regex

Note: you may need to restart the kernel to use updated packages.


In [33]:
from pathlib import Path
import os
import re

INPUT_DIR = Path("../data/raw/fcc")
OUTPUT_DIR = Path("../data/raw/fcc_sentencedelimited")

print("Current working dir:", os.getcwd())
p = Path(INPUT_DIR)
print("Raw path you gave:", INPUT_DIR)
print("Resolved (absolute) path:", p.resolve())
print("Exists?:", p.exists())
print("Is dir?:", p.is_dir())

# Try listing files (non-recursive)
try:
    files = list(p.iterdir())
    print("Number of entries:", len(files))
    for i, f in enumerate(files[:20]):
        print(i+1, f.name, " (dir)" if f.is_dir() else " (file)")
except Exception as e:
    print("Error listing directory:", repr(e))

Current working dir: /home/jen/TranslationsDataset/notebooks
Raw path you gave: ../data/raw/fcc
Resolved (absolute) path: /home/jen/TranslationsDataset/data/raw/fcc
Exists?: True
Is dir?: True
Number of entries: 21
1 Storm Surge Warning.csv  (file)
2 Earthquake Warning.csv  (file)
3 Fire Warning.csv  (file)
4 911 outage.csv  (file)
5 Dust Storm Warning.csv  (file)
6 Tornado Emergency.csv  (file)
7 fcc_alerts.xlsx  (file)
8 Hurricane Warning.csv  (file)
9 Tornado Warning.csv  (file)
10 Avalanche Warning.csv  (file)
11 Boil Water Advisory.csv  (file)
12 Flash Flood Warning.csv  (file)
13 Extreme Wind Warning.csv  (file)
14 Hazardous Materials Warning.csv  (file)
15 Snow Squall Warning.csv  (file)
16 WEA-Multilingual-Fillable-Alert-Templates-Guidance-2025.pdf  (file)
17 fcc_alerts_converted.xlsx  (file)
18 Severe Thunderstorm Warning.csv  (file)
19 Test Alert.csv  (file)
20 Tsunami Warning.csv  (file)


In [34]:
%pip install stanza

Note: you may need to restart the kernel to use updated packages.


## Helpers

In [35]:
import stanza
stanza.download('hi')   # Hindi model
stanza.download('zh')   # Chinese model
nlp_hi = stanza.Pipeline('hi', processors='tokenize', use_gpu=False, verbose=False)
nlp_zh = stanza.Pipeline('zh', processors='tokenize', use_gpu=False, verbose=False)

def split_hindi_stanza(text):
    doc = nlp_hi(text)
    return [''.join([t.text for t in s.tokens]) for s in doc.sentences]

def split_chinese_stanza(text):
    doc = nlp_zh(text)
    return [''.join([t.text for t in s.tokens]) for s in doc.sentences]

def is_excluded(col_name, exclude_list):
    if not exclude_list:
        return False
    low = col_name.lower()
    for exc in exclude_list:
        if exc.strip().lower() in low:
            return True
    return False


# ---------- stanza pipelines (lazy load) ----------
stanza_pipelines = {}

def get_stanza_pipeline(lang_code):
    # lang_code: 'zh' or 'hi'
    if lang_code in stanza_pipelines:
        return stanza_pipelines[lang_code]
    try:
        # Attempt to create pipeline; will download model if not present
        if lang_code == 'zh':
            stanza.download('zh', verbose=False)  # downloads if necessary
            nlp = stanza.Pipeline(lang='zh', processors='tokenize', use_gpu=False, verbose=False)
        elif lang_code == 'hi':
            stanza.download('hi', verbose=False)
            nlp = stanza.Pipeline(lang='hi', processors='tokenize', use_gpu=False, verbose=False)
        else:
            nlp = None
        stanza_pipelines[lang_code] = nlp
        return nlp
    except Exception as e:
        print(f"Warning: stanza pipeline not available for {lang_code}: {e}")
        stanza_pipelines[lang_code] = None
        return None

# ---------- splitters ----------
def split_chinese_stanza(text):
    nlp = get_stanza_pipeline('zh')
    if not nlp:
        return split_chinese_regex(text)
    doc = nlp(text)
    sents = []
    for sent in doc.sentences:
        sents.append(''.join([t.text for t in sent.tokens]).strip())
    return sents if sents else ([] if not text else [text.strip()])

def split_hindi_stanza(text):
    nlp = get_stanza_pipeline('hi')
    if not nlp:
        return split_hindi_regex(text)
    doc = nlp(text)
    sents = []
    for sent in doc.sentences:
        sents.append(''.join([t.text for t in sent.tokens]).strip())
    return sents if sents else ([] if not text else [text.strip()])

# fallback regex functions (as before) if stanza not available or fails
def split_chinese_regex(text):
    if not text: return []
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if any(ch in t for ch in "。！？"):
        parts = re.split(r'(?<=。|！|？)\s*', t)
        return [p.strip() for p in parts if p.strip()]
    if "如果" in t or "請" in t or "请" in t:
        parts = re.split(r'(?=(?:如果|如有|請|请))', t)
        return [p.strip() for p in parts if p.strip()]
    parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
    merged = []
    i = 0
    while i < len(parts):
        cur = parts[i]
        if len(cur) < 6 and i+1 < len(parts):
            cur = cur + '，' + parts[i+1]; i += 2
        else:
            i += 1
        merged.append(cur)
    return merged if merged else [t]

def split_hindi_regex(text):
    if not text: return []
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if '।' in t or re.search(r'[।\?\!\.]', t):
        parts = re.split(r'(?<=।|\?|!|\.)\s*', t)
        return [p.strip() for p in parts if p.strip()]
    if 'यदि' in t or 'अगर' in t:
        parts = re.split(r'(?=(?:यदि|अगर))', t); return [p.strip() for p in parts if p.strip()]
    parts = [p.strip() for p in re.split(r'[,\u0964;]\s*', t) if p.strip()]
    return parts if len(parts) > 1 else [t]

def split_generic_regex(text):
    if not text: return []
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if any(ch in t for ch in ['。','？','！']):
        parts = re.split(r'(?<=。|！|？)\s*', t); return [p.strip() for p in parts if p.strip()]
    if any(ch in t for ch in ['؟','؛','۔']):
        parts = re.split(r'(?<=\?|\u061F|;|؛)\s*', t); return [p.strip() for p in parts if p.strip()]
    if re.search(r'[\.!?]', t):
        parts = re.split(r'(?<=[\.!?])\s+', t); return [p.strip() for p in parts if p.strip()]
    if any(kw in t for kw in [' 如果 ', ' إذا ', ' If ', ' En cas ', ' Se ', ' Si ']):
        parts = re.split(r'(?=(?:如果|如果您|إذا|If|En cas|Se|Si)\b)', t); return [p.strip() for p in parts if p.strip()]
    parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
    return parts if len(parts) > 1 else [t]

def split_by_col(cell_text, col_name):
    hint = (col_name or "").lower()
    # prefer header hint
    if "simplified" in hint or "chinese" in hint or "zh" in hint:
        # prefer stanza if available
        try:
            return split_chinese_stanza(cell_text)
        except Exception:
            return split_chinese_regex(cell_text)
    if "hindi" in hint or "hi" == hint:
        try:
            return split_hindi_stanza(cell_text)
        except Exception:
            return split_hindi_regex(cell_text)
    # default
    return split_generic_regex(cell_text)

In [36]:
import re

# small helper to detect language by column name (prefer this if headers are reliable)
def lang_from_col(col_name):
    n = col_name.lower()
    if "chinese" in n or "simplified" in n or "zh" in n:
        return "zh"
    if "hindi" in n or "hi" in n:
        return "hi"
    # add other heuristics if needed
    return None

# Chinese-specific splitter (handles missing punctuation better)
def split_chinese(text):
    if not text: return []
    t = re.sub(r'\s+', ' ', text).strip()
    # First split on full stops/question/exclaim (normal)
    if any(ch in t for ch in "。！？"):
        parts = re.split(r'(?<=。|！|？)\s*', t)
        parts = [p.strip() for p in parts if p.strip()]
    else:
        # fallback: split at common clause starters or long commas, but keep reasonably sized segments
        # split on comma-like punctuation but then recombine very short fragments
        parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
        # recombine if pieces are tiny (<6 chars) with next piece to avoid over-splitting
        merged = []
        i = 0
        while i < len(parts):
            cur = parts[i]
            if len(cur) < 6 and i+1 < len(parts):
                cur = cur + '，' + parts[i+1]
                i += 2
            else:
                i += 1
            merged.append(cur)
        parts = merged
    return parts

# Hindi-specific splitter
def split_hindi(text):
    if not text: return []
    t = re.sub(r'\s+', ' ', text).strip()
    # Split on danda and punctuation (। ? ! .)
    if '।' in t or re.search(r'[।\?\!]', t):
        parts = re.split(r'(?<=।|\?|!|\.)\s*', t)
        parts = [p.strip() for p in parts if p.strip()]
    else:
        # split on commas and 'यदि' or 'अगर' as potential sentence starters
        if 'यदि' in t or 'अगर' in t:
            parts = re.split(r'(?=(?:यदि|अगर))', t)
            parts = [p.strip() for p in parts if p.strip()]
        else:
            parts = [p.strip() for p in re.split(r'[,\u0964;]\s*', t) if p.strip()]
    return parts

# Generic (fallback) splitter (keeps previous behavior)
def split_generic(text):
    if not isinstance(text, str) or not text.strip():
        return []
    t = text.strip()
    t = re.sub(r'\s+', ' ', t)

    # quick multi-script checks (CJK, arabic, latin)
    if any(ch in t for ch in ['。','？','！']):
        parts = re.split(r'(?<=。|！|？)\s*', t)
        return [p.strip() for p in parts if p.strip()]
    if any(ch in t for ch in ['؟','؛','۔']):
        parts = re.split(r'(?<=\?|\u061F|;|؛)\s*', t)
        return [p.strip() for p in parts if p.strip()]
    if re.search(r'[\.!?]', t):
        parts = re.split(r'(?<=[\.!?])\s+', t)
        return [p.strip() for p in parts if p.strip()]
    # fallback
    parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
    if len(parts) > 1:
        return parts
    return [t]

# Master dispatcher: pick splitter by language code or header hint
def split_sentences_by_lang(cell_text, col_name=None):
    # col_name is the CSV header for the column; prefer that to detect language
    lang_hint = lang_from_col(col_name) if col_name else None
    if lang_hint == 'zh':
        return split_chinese(cell_text)
    if lang_hint == 'hi':
        return split_hindi(cell_text)
    # fallback: try a cheap language detection if you have langdetect installed (optional)
    # from langdetect import detect
    # try:
    #     detected = detect(cell_text)
    # except:
    #     detected = None
    # if detected == 'zh-cn' or detected == 'zh':
    #     return split_chinese(cell_text)
    # if detected == 'hi':
    #     return split_hindi(cell_text)
    return split_generic(cell_text)

In [37]:
# Notebook cell: use stanza for Chinese and Hindi sentence splitting, align sentences across languages
import re
from pathlib import Path
import pandas as pd
import stanza

# ---------- CONFIG ----------
# Ensure INPUT_DIR and OUTPUT_DIR are defined beforehand in notebook, e.g.:
# INPUT_DIR = Path("/full/path/to/csv_folder")
# OUTPUT_DIR = Path("/full/path/to/analysis_out")
RECURSIVE = False
GLOB_PATTERN = "*.csv"
ENGLISH_COL = "English"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#exclude_langs = ["Hindi", "Chinese (Simplified)", "Chinese (Traditional)"]  # exclude these columns
exclude_langs = []  # exclude these columns

# ----------------------------

if 'INPUT_DIR' not in globals() or 'OUTPUT_DIR' not in globals():
    raise RuntimeError("Please define INPUT_DIR and OUTPUT_DIR Path variables first (e.g., INPUT_DIR = Path('...')).")

INPUT_DIR = Path(INPUT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

print("Input dir:", INPUT_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())



Input dir: /home/jen/TranslationsDataset/data/raw/fcc
Output dir: /home/jen/TranslationsDataset/data/raw/fcc_sentencedelimited


In [38]:

# -------- stanza pipelines (lazy) ----------
# download if necessary and create pipelines for zh and hi
stanza.download('zh', verbose=False)
stanza.download('hi', verbose=False)
nlp_zh = stanza.Pipeline(lang='zh', processors='tokenize', use_gpu=False, verbose=False)
nlp_hi = stanza.Pipeline(lang='hi', processors='tokenize', use_gpu=False, verbose=False)

# -------- helpers ----------
def col_is_excluded(col_name, exclude_list):
    if not exclude_list:
        return False
    low = col_name.lower()
    for exc in exclude_list:
        if exc.strip().lower() in low:
            return True
    return False

def split_chinese_stanza(text):
    if not text:
        return []
    doc = nlp_zh(text)
    return [''.join([t.text for t in s.tokens]).strip() for s in doc.sentences]

def split_hindi_stanza(text):
    if not text:
        return []
    doc = nlp_hi(text)
    return [''.join([t.text for t in s.tokens]).strip() for s in doc.sentences]

def split_generic_regex(text):
    if not text:
        return []
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if any(ch in t for ch in ['。','？','！']):
        parts = re.split(r'(?<=。|！|？)\s*', t)
        return [p.strip() for p in parts if p.strip()]
    if re.search(r'[\.!?]', t):
        parts = re.split(r'(?<=[\.!?])\s+', t)
        return [p.strip() for p in parts if p.strip()]
    parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
    return parts if len(parts) > 1 else [t]

def split_by_col(cell_text, col_name):
    cname = (col_name or "").lower()
    try:
        if "chinese" in cname or "simplified" in cname or "zh" in cname:
            return split_chinese_stanza(cell_text)
        if "hindi" in cname or cname.strip() == "hi":
            return split_hindi_stanza(cell_text)
    except Exception:
        pass
    return split_generic_regex(cell_text)

# -------- process CSVs --------
files = sorted(INPUT_DIR.rglob(GLOB_PATTERN) if RECURSIVE else INPUT_DIR.glob(GLOB_PATTERN))
files = [f for f in files if f.is_file()]
print(f"Found {len(files)} CSV files.")

if not files:
    raise RuntimeError("No CSV files found in INPUT_DIR.")

for csv_path in files:
    print("\nProcessing:", csv_path.name)
    try:
        df = pd.read_csv(csv_path, dtype=str).fillna("")
    except Exception as e:
        print("  ERROR reading file:", e)
        continue

    all_cols = list(df.columns)
    # Determine excluded columns (case-insensitive substring match)
    excluded = [c for c in all_cols if col_is_excluded(c, exclude_langs)]
    if excluded:
        print("  Excluding columns:", excluded)
    # Build output columns (English first), skipping excluded
    if ENGLISH_COL not in all_cols:
        print(f"  WARNING: English column '{ENGLISH_COL}' not found; skipping file.")
        continue
    if col_is_excluded(ENGLISH_COL, exclude_langs):
        print(f"  English column '{ENGLISH_COL}' is in exclude list. Skip file to avoid losing anchor.")
        continue

    out_cols = [ENGLISH_COL] + [c for c in all_cols if c != ENGLISH_COL and c not in excluded]
    print("  Output columns:", out_cols)

    aligned = []
    for _, row in df.iterrows():
        # split only the non-excluded columns
        percol = {c: split_by_col(str(row[c]), c) for c in out_cols}
        max_n = max((len(v) for v in percol.values()), default=0)
        for i in range(max_n):
            out = {}
            out[ENGLISH_COL] = percol[ENGLISH_COL][i] if i < len(percol[ENGLISH_COL]) else ""
            for c in out_cols:
                if c == ENGLISH_COL: continue
                out[c] = percol[c][i] if i < len(percol[c]) else ""
            aligned.append(out)

    out_df = pd.DataFrame(aligned, columns=out_cols)
    out_name = OUTPUT_DIR / f"{csv_path.stem}_aligned_by_sentence.csv"
    out_df.to_csv(out_name, index=False, encoding='utf-8')
    print("  wrote:", out_name.name, "rows:", len(out_df))

print("\nDone. Outputs in:", OUTPUT_DIR.resolve())

Found 18 CSV files.

Processing: 911 outage.csv
  Output columns: ['English', 'Arabic', 'Chinese (Simplified)', 'Chinese (Traditional)', 'French', 'German', 'Haitian Creole', 'Hindi', 'Italian', 'Korean', 'Portuguese', 'Russian', 'Spanish', 'Tagalog', 'Vietnamese']
  wrote: 911 outage_aligned_by_sentence.csv rows: 3

Processing: Avalanche Warning.csv
  Output columns: ['English', 'Arabic', 'Chinese (Simplified)', 'Chinese (Traditional)', 'French', 'German', 'Haitian Creole', 'Hindi', 'Italian', 'Korean', 'Portuguese', 'Russian', 'Spanish', 'Tagalog', 'Vietnamese']
  wrote: Avalanche Warning_aligned_by_sentence.csv rows: 9

Processing: Boil Water Advisory.csv
  Output columns: ['English', 'Arabic', 'Chinese (Simplified)', 'Chinese (Traditional)', 'French', 'German', 'Haitian Creole', 'Hindi', 'Italian', 'Korean', 'Portuguese', 'Russian', 'Spanish', 'Tagalog', 'Vietnamese']
  wrote: Boil Water Advisory_aligned_by_sentence.csv rows: 10

Processing: Dust Storm Warning.csv
  Output columns:

## test alignment for Chinese and Hindi

In [39]:
# Notebook cell: parse the provided CSV text with stanza for Chinese & Hindi and show aligned output
import re
from io import StringIO
import pandas as pd
import stanza
from pathlib import Path

# --- CSV text (exact sample you posted) ---
csv_text = r"""English,Arabic,Chinese (Simplified),Chinese (Traditional),French,German,Haitian Creole,Hindi,Italian,Korean,Portuguese,Russian,Spanish,Tagalog,Vietnamese
[SENDING AGENCY]: A 9-1-1 OUTAGE ALERT is in effect for [LOCATION].,[SENDING AGENCY]: يسري تنبيه انقطاع خدمات الطوارئ ( 1 - 1 - 9 ) في [LOCATION] حتى [TIME].,[SENDINGAGENCY]：[LOCATION]发布了9-1-1故障警报，当前生效中如果发生紧急情况，请直接拨打当地电话号码联系警察、消防、医疗或其他紧急服务部门。,[SENDINGAGENCY]：[LOCATION]發佈了9-1-1故障警報，當前生效中如果發生緊急情況，請直接撥打當地電話號碼聯絡警察、消防、醫療或其他緊急服務部門如果撥打9-1-1，您可能得不到幫助,[SENDING AGENCY]: un AVIS D’INTERRUPTION DU 9-1-1 est en vigueur pour [LOCATION].,[SENDING AGENCY] Ein 9-1-1-AUSFALLALARM ist für [LOCATION] in Kraft.,[SENDING AGENCY]: Yon AVÈTISMAN ENTÈRIPSYON 9-1-1 nan zòn sa a [LOCATION].,[SENDINGAGENCY]:[LOCATION]में9-1-1सेवामेंअवरोधकाअलर्टप्रभावीहै।,[SENDING AGENCY]: Un AVVISO DI INTERRUZIONE del servizio 9-1-1 è in vigore per [LOCATION].,[SENDING AGENCY]: [LOCATION]에 911 비상 경보가 발효 중입니다.,[SENDING AGENCY]: Um ALERTA DE INTERRUPÇÃO DO SERVIÇO 9-1-1 está em vigor para [LOCATION].,[SENDING AGENCY]: В [LOCATION] НЕДОСТУПЕН НОМЕР 9-1-1.,[SENDING AGENCY]: UNA ALERTA POR INTERRUPCIÓN DE LOS SERVICIOS 9-1-1 está en efecto en [LOCATION].,[SENDING AGENCY]: May bisa ang ALERT SA PAGKWALA NG 9-1-1 sa [LOCATION] hanggang [TIME].,[SENDING AGENCY]: CẢNH BÁO SẬP NGUỒN 9-1-1 có hiệu lực tại [LOCATION].
"Please contact police, fire, medical, or other emergency services directly at their local phone numbers in case of emergency.",يُرجى الاتصال بالشرطة أو الإطفاء أو الخدمات الطبية أو غيرها من خدمات الطوارئ مباشرة على أرقام هواتفهم المحلية في حالات الطوارئ.,如果您拨打9-1-1，则可能无法获得帮助,如果發生緊急情況，,"En cas d'urgence, veuillez contacter la police, les pompiers, les services médicaux ou tout autre service d'urgence directement en appelant leur numéro de téléphone local.","Bitte wenden Sie sich in Notfällen direkt an die Polizei, die Feuerwehr, die Rettungsdienste oder andere Notdienste unter den jeweiligen örtlichen Telefonnummern.","Tanpri kontakte lapolis, ponpye, medikal, oswa lòt sèvis ijans dirèkteman nan nimewo telefòn lokal yo nan ka ijans.","आपातस्थिततहोनेपरकृपयापुतलस,दमकल,तितकत्सायाअन्यआपातसेवाओंसेसीधेउनकेथिानीयफोननंबरपरसंपकटकरें।","In caso di emergenza, contattare la Polizia, i Vigili del Fuoco, i servizi medici o altri servizi di emergenza direttamente ai numeri di telefono locali.","긴급 상황 발생 시 해당 지역 전화번호로 경찰서, 소방서, 의료 또는 기타 응급 서비스 기관에 직접 연락하십시오.","Em caso de emergência, entre em contato com a polícia, bombeiros, médicos ou outros serviços de emergência diretamente por meio dos seus números de telefone locais.","При возникновении чрезвычайной ситуации вызывайте полицию, пожарных, скорую помощь или другие экстренные службы по местным телефонным номерам.","En caso de emergencia, comuníquese directamente con la policía, los bomberos, los servicios médicos u otros servicios de emergencia, llamando a sus números locales.","Pakikontak ang pulisya, bumbero, medikal, o iba pang mga serbisyong pang-emergency nang direkta sa kanilang mga lokal na numero ng telepono kung sakaling magkaroon ng emergency.","Vui lòng liên hệ trực tiếp số điện thoại địa phương với cảnh sát, cứu hỏa, y tế, hoặc các dịch vụ cấp cứu khác trong trường hợp khẩn cấp."
"If you dial 9-1-1, you may not get help.",إذا اتصلت بالرقم 1 - 1 - 9 ، فقد لا تحصل على المساعدة .,,,"Si vous composez le 9-1-1, vous risquez de ne pas obtenir d'aide.","Wenn Sie 9-1-1 wählen, erhalten Sie möglicherweise keine Hilfe.","Si w rele 9-1-1, ou ka pa jwenn èd.","अगरआप9-1-1डायलकरें,तोहोसकताहैतकआपकोमददनतमले।",Il 9 -1-1 potrebbe non essere disponibile in caso di emergenza.,911에 전화해도 도움을 받지 못할 수도 있습니다.,É possível você não conseguir ajuda ao discar 9-1-1.,"Если вы наберете 9-1-1, помощь может не прибыть.","Si llama al 9-1-1, es posible que no reciba ayuda.","Kung magda-dial ka sa 9-1-1, maaaring hindi ka makakuha ng tulong.",Nếu gọi số 9-1-1 có thể bạn không được trợ giúp.
"""

# ---------- load CSV into DataFrame ----------
df = pd.read_csv(StringIO(csv_text), dtype=str).fillna("")

# ---------- prepare stanza pipelines ----------
# download & initialize zh and hi if not already
print("Loading/Downloading stanza models if necessary...")
stanza.download('zh', verbose=False)
stanza.download('hi', verbose=False)
nlp_zh = stanza.Pipeline(lang='zh', processors='tokenize', use_gpu=False, verbose=False)
nlp_hi = stanza.Pipeline(lang='hi', processors='tokenize', use_gpu=False, verbose=False)
print("Stanza pipelines ready.")

# ---------- define split functions that use stanza ----------
def split_chinese_stanza(text):
    if not text: 
        return []
    doc = nlp_zh(text)
    sents = []
    for sent in doc.sentences:
        sents.append(''.join([tok.text for tok in sent.tokens]).strip())
    return sents

def split_hindi_stanza(text):
    if not text:
        return []
    doc = nlp_hi(text)
    sents = []
    for sent in doc.sentences:
        sents.append(''.join([tok.text for tok in sent.tokens]).strip())
    return sents

# generic fallback (as used earlier)
def split_generic(text):
    if not text:
        return []
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if any(ch in t for ch in ['。','？','！']):
        parts = re.split(r'(?<=。|！|？)\s*', t); return [p.strip() for p in parts if p.strip()]
    if re.search(r'[\.!?]', t):
        parts = re.split(r'(?<=[\.!?])\s+', t); return [p.strip() for p in parts if p.strip()]
    parts = [p.strip() for p in re.split(r'[，,；;]\s*', t) if p.strip()]
    return parts if len(parts)>1 else [t]

def split_by_col(cell_text, col_name):
    cn = col_name.lower()
    if "chinese" in cn or "simplified" in cn or "zh" in cn:
        try:
            return split_chinese_stanza(cell_text)
        except Exception:
            return split_generic(cell_text)
    if "hindi" in cn or "hi" in cn:
        try:
            return split_hindi_stanza(cell_text)
        except Exception:
            return split_generic(cell_text)
    return split_generic(cell_text)

# ---------- align sentences ----------
cols = list(df.columns)
EN = "English"
other_cols = [c for c in cols if c != EN]

aligned = []
for _, row in df.iterrows():
    percol = {c: split_by_col(str(row[c]), c) for c in cols}
    max_n = max(len(v) for v in percol.values()) if percol else 0
    for i in range(max_n):
        out = {}
        out[EN] = percol[EN][i] if i < len(percol[EN]) else ""
        for c in other_cols:
            out[c] = percol[c][i] if i < len(percol[c]) else ""
        aligned.append(out)

out_df = pd.DataFrame(aligned, columns=[EN] + other_cols)

# display result
out_df

Loading/Downloading stanza models if necessary...
Stanza pipelines ready.


,English,Arabic,Chinese (Simplified),Chinese (Traditional),French,German,Haitian Creole,Hindi,Italian,Korean,Portuguese,Russian,Spanish,Tagalog,Vietnamese
0,[SENDING AGENCY]: A 9-1-1 OUTAGE ALERT is in e...,[SENDING AGENCY]: يسري تنبيه انقطاع خدمات الطو...,[SENDINGAGENCY]：[LOCATION]发布了9-1-1故障警报，当前生效中如果...,[SENDINGAGENCY]：[LOCATION]發佈了9-1-1故障警報，當前生效中如果...,[SENDING AGENCY]: un AVIS D’INTERRUPTION DU 9-...,[SENDING AGENCY] Ein 9-1-1-AUSFALLALARM ist fü...,[SENDING AGENCY]: Yon AVÈTISMAN ENTÈRIPSYON 9-...,[SENDINGAGENCY]:[LOCATION]में9-1-1सेवामेंअवरोध...,[SENDING AGENCY]: Un AVVISO DI INTERRUZIONE de...,[SENDING AGENCY]: [LOCATION]에 911 비상 경보가 발효 중입니다.,[SENDING AGENCY]: Um ALERTA DE INTERRUPÇÃO DO ...,[SENDING AGENCY]: В [LOCATION] НЕДОСТУПЕН НОМЕ...,[SENDING AGENCY]: UNA ALERTA POR INTERRUPCIÓN ...,[SENDING AGENCY]: May bisa ang ALERT SA PAGKWA...,[SENDING AGENCY]: CẢNH BÁO SẬP NGUỒN 9-1-1 có ...
1,,,,如果撥打9-1-1，您可能得不到幫助,,,,,,,,,,,
2,"Please contact police, fire, medical, or other...",يُرجى الاتصال بالشرطة أو الإطفاء أو الخدمات ال...,如果您拨打9-1-1，则可能无法获得帮助,如果發生緊急情況，,"En cas d'urgence, veuillez contacter la police...",Bitte wenden Sie sich in Notfällen direkt an d...,"Tanpri kontakte lapolis, ponpye, medikal, oswa...","आपातस्थिततहोनेपरकृपयापुतलस,दमकल,तितकत्सायाअन्य...","In caso di emergenza, contattare la Polizia, i...","긴급 상황 발생 시 해당 지역 전화번호로 경찰서, 소방서, 의료 또는 기타 응급 서...","Em caso de emergência, entre em contato com a ...",При возникновении чрезвычайной ситуации вызыва...,"En caso de emergencia, comuníquese directament...","Pakikontak ang pulisya, bumbero, medikal, o ib...",Vui lòng liên hệ trực tiếp số điện thoại địa p...
3,"If you dial 9-1-1, you may not get help.",إذا اتصلت بالرقم 1 - 1 - 9 ، فقد لا تحصل على ا...,,,"Si vous composez le 9-1-1, vous risquez de ne ...","Wenn Sie 9-1-1 wählen, erhalten Sie möglicherw...","Si w rele 9-1-1, ou ka pa jwenn èd.","अगरआप9-1-1डायलकरें,तोहोसकताहैतकआपकोमददनतमले।",Il 9 -1-1 potrebbe non essere disponibile in c...,911에 전화해도 도움을 받지 못할 수도 있습니다.,É possível você não conseguir ajuda ao discar ...,"Если вы наберете 9-1-1, помощь может не прибыть.","Si llama al 9-1-1, es posible que no reciba ay...","Kung magda-dial ka sa 9-1-1, maaaring hindi ka...",Nếu gọi số 9-1-1 có thể bạn không được trợ giúp.
